# Indian Document Understanding Tool: Colab runner
Patram-7B + IndicTrans2 + page retrieval.

**Before you start**
1. Runtime → Change runtime type → **GPU** (T4 is fine).
2. Request access to https://huggingface.co/bharatgenai/patram-7b-instruct and create a *read* token.
3. Add the token as a Colab secret named `HF_TOKEN` (key icon in the sidebar, *Notebook access* on).

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 1. Get the code
Use **Option A** (GitHub) or **Option B** (upload the zip).

In [ ]:
# Option A: clone from GitHub (set your repo URL)
REPO_URL = "https://github.com/<your-username>/indian-doc-understanding.git"
!git clone {REPO_URL} /content/indian-doc-understanding
%cd /content/indian-doc-understanding

In [ ]:
# Option B: upload indian-doc-understanding.zip
import zipfile
from google.colab import files
up = files.upload()
zipfile.ZipFile(next(iter(up))).extractall("/content")
%cd /content/indian-doc-understanding

## 2. Install (OCR packs + Python packages)

In [ ]:
!apt-get -qq install -y tesseract-ocr tesseract-ocr-hin tesseract-ocr-mar > /dev/null
!pip install -q -r requirements.txt
# If a tokenizer / transformers error appears later: Runtime → Restart session, then re-run from section 3.

## 3. Log in to Hugging Face and run the unit tests

In [ ]:
import sys; sys.path.insert(0, "src")
from idu.models import hf_login
print("HF login:", hf_login())

In [ ]:
!python -m pytest -q tests

## 4. Launch the app
Opens a public Gradio link (valid ~1 week). Stop this cell (■) before running the evaluation: both load the models.

In [ ]:
!python app.py --share

## 5. Evaluate
Runs the test cases in `eval/testcases.jsonl`. Run all three for the ablation table in your report.

In [ ]:
!python eval/run_eval.py --out eval/results/run1
!python eval/run_eval.py --out eval/results/run1_goldpage --gold-page
!python eval/run_eval.py --out eval/results/run1_noverify --no-verify

In [ ]:
from IPython.display import Markdown
Markdown(open("eval/results/run1/failure_analysis.md", encoding="utf-8").read())

In [ ]:
!zip -qr results.zip eval/results
from google.colab import files
files.download("results.zip")